# Eval Results Analysis (remote)


In [12]:
import json
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from huggingface_hub import HfApi
from IPython.display import display

load_dotenv()

HF_ENDPOINT = "https://huggingface.co".rstrip("/")
# HF_TOKEN = os.environ.get("HF_TOKEN")

REPO_ID = "anon-review-12345/dora-reproducibility-study"
REVISION = "main"

RUNS = {
    "qdora-seed42-fp32":  {"repo_dir": "qdora-fp32/seed-42",    "label": "QDoRA (seed 42) - fp32"},
    "qdora-seed43-fp32":  {"repo_dir": "qdora-fp32/seed-43",    "label": "QDoRA (seed 43) - fp32"},
    "lora-v3":            {"repo_dir": "lora/seed-42",          "label": "LoRA (seed 42)"},
    "lora-match-seed42":  {"repo_dir": "lora-lr-match/seed-42", "label": "LoRA LR-Match (seed 42)"},
    "lora-match-seed43":  {"repo_dir": "lora-lr-match/seed-43", "label": "LoRA LR-Match (seed 43)"},
    "dora-seed42":        {"repo_dir": "dora/seed-42",          "label": "DoRA (seed 42)"},
    "dora-seed43":        {"repo_dir": "dora/seed-43",          "label": "DoRA (seed 43)"},
    "qlora-seed42":       {"repo_dir": "qlora/seed-42",         "label": "QLoRA (seed 42)"},
    "qlora-seed43":       {"repo_dir": "qlora/seed-43",         "label": "QLoRA (seed 43)"},
}

# Same task order as eval.py
DEFAULT_TASKS = [
    "boolq",
    "piqa",
    "social_i_qa",
    "ARC-Challenge",
    "ARC-Easy",
    "openbookqa",
    "hellaswag",
    "winogrande",
]

ANSWER_PATTERNS = {
    "boolq": r"true|false",
    "piqa": r"solution1|solution2",
    "social_i_qa": r"answer1|answer2|answer3|answer4|answer5",
    "ARC-Challenge": r"answer1|answer2|answer3|answer4|answer5",
    "ARC-Easy": r"answer1|answer2|answer3|answer4|answer5",
    "openbookqa": r"answer1|answer2|answer3|answer4|answer5",
    "hellaswag": r"ending1|ending2|ending3|ending4",
    "winogrande": r"option1|option2",
}

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Remote file access

The same primitives used by `mechanistic_analysis_remote.ipynb`. Everything this
notebook reads is small JSON, so no byte-range machinery is needed here.

In [13]:
SESSION = requests.Session()
# if HF_TOKEN:
#     SESSION.headers["Authorization"] = f"Bearer {HF_TOKEN}"

REQUEST_TIMEOUT = 60
REQUEST_RETRIES = 3


def http_get(repo_id, path, revision=REVISION, headers=None):
    url = f"{HF_ENDPOINT}/{repo_id}/resolve/{revision}/{path}"
    last_error = None
    for _ in range(REQUEST_RETRIES):
        try:
            response = SESSION.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response
        except requests.RequestException as error:
            last_error = error
    raise RuntimeError(f"GET failed after {REQUEST_RETRIES} attempts: {url}") from last_error


def read_remote_json(repo_id, path, revision=REVISION):
    return json.loads(http_get(repo_id, path, revision).text)


HF_API = HfApi()

try:
    REPO_FILES = set(HF_API.list_repo_files(REPO_ID, revision=REVISION))
except Exception as error:
    raise RuntimeError(
        f"Could not list {REPO_ID}@{REVISION}. Gated repositories need a valid "
        "HF_TOKEN in the environment or in .env."
    ) from error

print(f"{REPO_ID}@{REVISION}: {len(REPO_FILES):,} files listed (no download yet)")

anon-review-12345/dora-reproducibility-study@main: 797 files listed (no download yet)


## Resolve one eval run per entry

`paper_eval_<timestamp>` directory names sort by recency, so the newest one wins.
Every entry is validated against the file listing before anything is downloaded.

In [14]:
resolved_runs = {}
problems = []

for run_id, spec in RUNS.items():
    prefix = f"{spec['repo_dir']}/paper_eval_results/"
    candidates = sorted({
        path[len(prefix):].split("/")[0]
        for path in REPO_FILES
        if path.startswith(prefix) and "/" in path[len(prefix):]
    })
    if not candidates:
        problems.append(f"No paper_eval run under {REPO_ID}/{prefix}")
        continue

    eval_dir = prefix + candidates[-1]  # naming standard sorts by recency
    has_predictions = any(
        path.startswith(f"{eval_dir}/predictions/") for path in REPO_FILES
    )
    if f"{eval_dir}/results.json" not in REPO_FILES and not has_predictions:
        problems.append(f"Missing both predictions/ and results.json: {eval_dir}")
        continue

    resolved_runs[run_id] = {"eval_dir": eval_dir, "label": spec["label"]}
    extra = f" (newest of {len(candidates)}: {', '.join(candidates)})" if len(candidates) > 1 else ""
    print(f"{run_id:>18} -> {eval_dir}{extra}")

if problems:
    details = "\n  - ".join(problems)
    raise ValueError(f"Invalid RUNS entries:\n  - {details}")

print(f"\nUsing {len(resolved_runs)} run(s)")

 qdora-seed42-fp32 -> qdora-fp32/seed-42/paper_eval_results/paper_eval_20260825_015812
 qdora-seed43-fp32 -> qdora-fp32/seed-43/paper_eval_results/paper_eval_20260825_015842
           lora-v3 -> lora/seed-42/paper_eval_results/paper_eval_20260827_103908
 lora-match-seed42 -> lora-lr-match/seed-42/paper_eval_results/paper_eval_20260717_002852 (newest of 2: paper_eval_20260712_201111, paper_eval_20260717_002852)
 lora-match-seed43 -> lora-lr-match/seed-43/paper_eval_results/paper_eval_20260820_194824
       dora-seed42 -> dora/seed-42/paper_eval_results/paper_eval_20260827_022522 (newest of 2: paper_eval_20260710_192428, paper_eval_20260827_022522)
       dora-seed43 -> dora/seed-43/paper_eval_results/paper_eval_20260722_130811
      qlora-seed42 -> qlora/seed-42/paper_eval_results/paper_eval_20260814_114804
      qlora-seed43 -> qlora/seed-43/paper_eval_results/paper_eval_20260721_213142

Using 9 run(s)


## Per-run accuracy

In [ ]:
runs = []
for run_id, spec in resolved_runs.items():
    eval_dir = spec["eval_dir"]
    task_data = {}

    results_path = f"{eval_dir}/results.json"
    if results_path in REPO_FILES:
        payload = read_remote_json(REPO_ID, results_path)
        for task, result in payload.get("results", {}).items():
            task_data[task] = {
                "accuracy": result.get("accuracy"),
                "correct": result.get("correct"),
                "total": result.get("total"),
                "parse_failures": result.get("parse_failures"),
            }

    # Also scan predictions/ - picks up tasks that finished after the last
    # results.json checkpoint, or runs that never wrote a results.json. These
    # files are large, so only the ones results.json does not cover are fetched.
    predictions_prefix = f"{eval_dir}/predictions/"
    prediction_files = sorted(
        path for path in REPO_FILES
        if path.startswith(predictions_prefix) and path.endswith(".json")
    )
    for path in prediction_files:
        task = path[len(predictions_prefix):-len(".json")]
        if task in task_data and task_data[task]["accuracy"] is not None:
            continue  # trust results.json

        print(f"  {run_id}: {task} missing from results.json, reading predictions")
        predictions = read_remote_json(REPO_ID, path)
        correct = sum(1 for row in predictions if row.get("correct"))
        parse_failures = sum(1 for row in predictions if row.get("pred", "") == "")
        total = len(predictions)
        task_data[task] = {
            "accuracy": correct / total if total else 0.0,
            "correct": correct,
            "total": total,
            "parse_failures": parse_failures,
        }

    runs.append({
        "model": spec["label"],
        "run": eval_dir.rsplit("/", 1)[-1],
        "run_dir": eval_dir,
        "tasks": task_data,
    })

print(f"Collected {len(runs)} run(s), {sum(len(run['tasks']) for run in runs)} task results")

## Build the results table

One row per run, one column per benchmark, plus `average` = mean of the
available task accuracies for that run. `run_dir` is the path inside the Hub
repo, which the drill-down section below reads back.

In [ ]:
rows = []
for run in runs:
    row = {"model": run["model"], "run": run["run"]}
    for task in DEFAULT_TASKS:
        entry = run["tasks"].get(task)
        row[task] = entry["accuracy"] if entry else float("nan")
    # Include any extra tasks not in the default list
    for task, entry in run["tasks"].items():
        if task not in DEFAULT_TASKS:
            row[task] = entry["accuracy"]
    row["run_dir"] = run["run_dir"]
    rows.append(row)

results_df = pd.DataFrame(rows)
task_columns = [
    column for column in results_df.columns
    if column not in {"model", "run", "run_dir"}
]
results_df["average"] = results_df[task_columns].mean(axis=1, skipna=True)

# Column order: identifiers, tasks (in DEFAULT_TASKS order first), average, run_dir
results_df = results_df[
    ["model", "run"]
    + [column for column in DEFAULT_TASKS if column in results_df.columns]
    + [column for column in task_columns if column not in DEFAULT_TASKS]
    + ["average", "run_dir"]
].reset_index(drop=True)

results_df

## Counts table (correct / total per task)

In [ ]:
count_rows = []
for run in runs:
    row = {"model": run["model"], "run": run["run"]}
    for task in DEFAULT_TASKS:
        entry = run["tasks"].get(task)
        row[task] = f"{entry['correct']}/{entry['total']}" if entry else ""
    count_rows.append(row)

counts_df = pd.DataFrame(count_rows)
counts_df

## Parse-failure check

In [ ]:
pf_rows = []
for run in runs:
    row = {"model": run["model"], "run": run["run"]}
    for task in DEFAULT_TASKS:
        entry = run["tasks"].get(task)
        if entry and entry["total"]:
            row[task] = entry["parse_failures"] / entry["total"]
        else:
            row[task] = float("nan")
    pf_rows.append(row)

parse_fail_df = pd.DataFrame(pf_rows)
parse_fail_df.style.format(
    {c: "{:.2%}" for c in parse_fail_df.columns if c not in {"model", "run"}}
).map(
    lambda v: "background-color: #ffcccc" if isinstance(v, float) and v > 0.05 else "",
    subset=[c for c in parse_fail_df.columns if c not in {"model", "run"}],
)

## Export

In [ ]:
OUT_CSV = OUTPUT_DIR / "eval_results_table.csv"
results_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV.resolve()}")

## Drill into a specific run

This is the one place that downloads a full predictions file (a few hundred KB
to a few MB), so it only runs for the single run/task selected here.

In [ ]:
RUN_INDEX = 0  # row in results_df
TASK = "boolq"

run_dir = results_df.iloc[RUN_INDEX]["run_dir"]
predictions_path = f"{run_dir}/predictions/{TASK}.json"

if predictions_path in REPO_FILES:
    preds_df = pd.DataFrame(read_remote_json(REPO_ID, predictions_path))
    columns = [
        column for column in ["gold", "pred", "correct", "output_pred"]
        if column in preds_df.columns
    ]
    print(f"{len(preds_df)} examples - first 10:")
    display(preds_df[columns].head(10))

    print("\nParse failures (pred == ''):")
    display(preds_df[preds_df["pred"] == ""][columns].head(10))
else:
    print(f"No predictions file at {REPO_ID}/{predictions_path}")